# **Random Forest — Projet Plantes**

**But :** entraîner et évaluer un modèle **Random Forest** sur les deux bases du projet, en respectant le protocole ML commun.

**Bases :** TAXONS et MALADIES.  
**Représentations :** pixels bruts (32×32) et HOG.  
**Protocole :** mêmes splits train/val/test et mêmes *class weights* que les autres modèles ML.

---


Ce notebook constitue la partie **Random Forest** de la comparaison des modèles de **Machine Learning **.

Les expériences sont réalisées sur les deux représentations communes :

- **Pixels bruts** : images redimensionnées en 32×32 puis vectorisées ;
- **HOG** : descripteurs *Histogram of Oriented Gradients*.

Les données et les splits sont ceux définis dans le **Protocole ML commun**, afin que les performances soient comparables entre les quatre modèles évalués.

# Classification TAXONS

## Random Forest sur les pixels

In [ ]:
# ============================================================
# 1. VÉRIFICATION DE L'ACCÈS — à lancer en premier
# ============================================================
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

BASE = Path("/content/drive/MyDrive/Projet_Plantes")

if not BASE.exists():
    print("❌ Projet_Plantes INTROUVABLE dans MyDrive.\n")
    print("Le dossier est en 'Partagés avec moi' : Colab ne le voit pas.")
    print("Corrige en 10 secondes :")
    print("   1. Google Drive → Partagés avec moi")
    print("   2. Clic droit sur 'Projet_Plantes'")
    print("   3. Organiser → Ajouter un raccourci")
    print("   4. Choisis 'Mon Drive' → Ajouter")
    print("   5. Relance cette cellule\n")
    raise SystemExit("Raccourci Drive manquant — voir instructions ci-dessus.")

print("✅ Projet_Plantes accessible")
print("Contenu :", sorted(p.name for p in BASE.iterdir())[:10])

Mounted at /content/drive
✅ Projet_Plantes accessible
Contenu : ['Bronislaw', 'Ghislain', 'MALADIES.zip', 'Messaline', 'Modèles CNN', 'Paul el Forzli', 'README.txt', 'README2.txt', 'Rapport Modélisation', 'TAXONS.zip']


In [ ]:
# ============================================================
# CHARGEMENT DONNEES
# ============================================================
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

FEAT_DIR = BASE / "features" / "taxons_raw"
tr = np.load(FEAT_DIR / "train.npz", allow_pickle=True)
te = np.load(FEAT_DIR / "test.npz",  allow_pickle=True)
val = np.load(FEAT_DIR / "val.npz", allow_pickle=True)

X_tr_px_taxons, y_tr_str = tr["X"], tr["y"]
X_te_px_taxons, y_te_str = te["X"], te["y"]
X_val_px_taxons, y_val_str = val["X"], val["y"]

le_px_taxons = LabelEncoder().fit(y_tr_str)

y_tr_px_taxons = le_px_taxons.transform(y_tr_str)
y_te_px_taxons = le_px_taxons.transform(y_te_str)
y_val_px_taxons = le_px_taxons.transform(y_val_str)



print("Train X :", X_tr_px_taxons.shape)
print("Val X   :", X_val_px_taxons.shape)
print("Test X  :", X_te_px_taxons.shape)

print("Classes :", len(le_px_taxons.classes_))
print(le_px_taxons.classes_[:10])

Train X : (67377, 3072)
Val X   : (14370, 3072)
Test X  : (14418, 3072)
Classes : 14
['apple' 'blueberry' 'cherry_including_sour' 'grape' 'maize' 'orange'
 'peach' 'pepper_bell' 'potato' 'raspberry']


In [ ]:
param_grid = {
    "n_estimators": [50, 100],
    "max_features": ["sqrt"],
    "max_depth": [10, 20]}

from sklearn.model_selection import GridSearchCV

grid_pixels = GridSearchCV(
    estimator=rf_pixels,
    param_grid=param_grid,
    cv=3,
    scoring="f1_weighted",
    verbose=2,
    n_jobs=-1)

grid_pixels.fit(X_tr_px_taxons, y_tr_px_taxons)

In [ ]:
best_rf_pixels = grid_pixels.best_estimator_

print("Best params :", grid_pixels.best_params_)
print("Best CV score :", grid_pixels.best_score_)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns


# Prédictions
y_train_pred_taxons = best_rf_pixels.predict(X_tr_px_taxons)
y_test_pred_taxons  = best_rf_pixels.predict(X_te_px_taxons)
y_val_pred_taxons  = best_rf_pixels.predict(X_val_px_taxons)


# Scores
print("TRAIN accuracy :", accuracy_score(y_tr_px_taxons, y_train_pred_taxons))
print("VAL accuracy  :", accuracy_score(y_val_px_taxons, y_val_pred_taxons))
print("TEST accuracy  :", accuracy_score(y_te_px_taxons, y_test_pred_taxons))

print("TRAIN F1 :", f1_score(y_tr_px_taxons, y_train_pred_taxons, average="weighted"))
print("VAL F1  :", f1_score(y_val_px_taxons, y_val_pred_taxons, average="weighted"))
print("TEST F1  :", f1_score(y_te_px_taxons, y_test_pred_taxons, average="weighted"))


# Classification report
print("\n=== CLASSIFICATION REPORT TEST ===\n")

print(classification_report(
    y_te_px_taxons,
    y_test_pred_taxons,
    target_names=le_px_taxons.classes_
))


# Matrice de confusion
cm = confusion_matrix(
    y_te_px_taxons,
    y_test_pred_taxons
)

plt.figure(figsize=(12, 10))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=le_px_taxons.classes_,
    yticklabels=le_px_taxons.classes_
)

plt.xlabel("Prédit")
plt.ylabel("Réel")
plt.title("Matrice de confusion - Random Forest pixels")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## Random Forest sur HOG


In [ ]:
FEAT_DIR = BASE / "features" / "taxons_hog"

tr = np.load(FEAT_DIR / "train.npz", allow_pickle=True)
te = np.load(FEAT_DIR / "test.npz", allow_pickle=True)
val = np.load(FEAT_DIR / "val.npz", allow_pickle=True)

X_tr_hog_taxons, y_tr_str = tr['X'], tr['y']
X_te_hog_taxons, y_te_str = te['X'], te['y']
X_val_hog_taxons, y_val_str = val["X"], val["y"]

le_hog_taxons = LabelEncoder().fit(y_tr_str)

y_tr_hog_taxons = le_hog_taxons.transform(y_tr_str)
y_te_hog_taxons = le_hog_taxons.transform(y_te_str)
y_val_hog_taxons = le_hog_taxons.transform(y_val_str)

print("Train HOG :", X_tr_hog_taxons.shape)
print("Val HOG   :", X_val_hog_taxons.shape)
print("Test HOG  :", X_te_hog_taxons.shape)
print("Classes :", len(le_hog_taxons.classes_))

In [ ]:
from sklearn.model_selection import GridSearchCV


rf_hog = RandomForestClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1)


param_grid_hog = {
    "n_estimators": [50,100],
    "max_features": ["sqrt"],
    "max_depth": [10,20]
}


grid_hog = GridSearchCV(
    estimator=rf_hog,
    param_grid=param_grid_hog,
    cv=3,
    scoring="f1_weighted",
    verbose=2,
    n_jobs=-1
)


grid_hog.fit(
    X_tr_hog_taxons,
    y_tr_hog_taxons
)


best_rf_hog = grid_hog.best_estimator_

print("Best params :", grid_hog.best_params_)
print("Best CV score :", grid_hog.best_score_)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns


y_train_pred_hog = best_rf_hog.predict(X_tr_hog_taxons)
y_val_pred_hog   = best_rf_hog.predict(X_val_hog_taxons)
y_test_pred_hog  = best_rf_hog.predict(X_te_hog_taxons)

print("TRAIN accuracy :",
      accuracy_score(y_tr_hog_taxons, y_train_pred_hog))
print("VAL accuracy :",
      accuracy_score(y_val_hog_taxons, y_val_pred_hog))
print("TEST accuracy :",
      accuracy_score(y_te_hog_taxons, y_test_pred_hog))

print("TRAIN F1 :",
      f1_score(y_tr_hog_taxons, y_train_pred_hog, average="weighted"))
print("VAL F1 :",
      f1_score(y_val_hog_taxons, y_val_pred_hog, average="weighted"))
print("TEST F1 :",
      f1_score(y_te_hog_taxons, y_test_pred_hog, average="weighted"))


In [ ]:
print(classification_report(
    y_te_hog_taxons,
    y_test_pred_hog,
    target_names=le_hog_taxons.classes_
))


cm_hog = confusion_matrix(
    y_te_hog_taxons,
    y_test_pred_hog
)


plt.figure(figsize=(12,10))

sns.heatmap(
    cm_hog,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=le_hog_taxons.classes_,
    yticklabels=le_hog_taxons.classes_
)

plt.xlabel("Prédit")
plt.ylabel("Réel")
plt.title("Matrice de confusion - Random Forest HOG")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SCORES PIXELS TAXONS
# ============================================================

acc_pixels = accuracy_score(
    y_te_px_taxons,
    y_test_pred_taxons
)

f1_pixels = f1_score(
    y_te_px_taxons,
    y_test_pred_taxons,
    average="weighted"
)


# ============================================================
# SCORES HOG TAXONS
# ============================================================

acc_hog = accuracy_score(
    y_te_hog_taxons,
    y_test_pred_hog
)

f1_hog = f1_score(
    y_te_hog_taxons,
    y_test_pred_hog,
    average="weighted"
)


print("Pixels - Accuracy :", acc_pixels)
print("Pixels - F1 :", f1_pixels)

print("HOG - Accuracy :", acc_hog)
print("HOG - F1 :", f1_hog)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Données
methods = ["Pixels", "HOG"]

accuracy = [
    acc_pixels,
    acc_hog]

f1 = [
    f1_pixels,
    f1_hog]

# Position des barres
x = np.arange(len(methods))
width = 0.35

plt.figure(figsize=(8,5))

# Accuracy en bleu
bars1 = plt.bar(
    x - width/2,
    accuracy,
    width,
    color="steelblue",
    label="Accuracy")

# F1 en orange
bars2 = plt.bar(
    x + width/2,
    f1,
    width,
    color="lightblue",
    label="F1 weighted")

# Affichage des valeurs au-dessus des barres
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width()/2,
            height + 0.01,
            f"{height:.3f}",
            ha="center",
            fontsize=10)

plt.style.use("default")

plt.xticks(x, methods)
plt.ylabel("Score")
plt.ylim(0, 1)

plt.title("Comparaison Random Forest - Pixels vs HOG")

plt.legend()
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

# Classification Maladies

## Random Forest sur les pixels

In [ ]:
# ============================================================
# 1. VÉRIFICATION DE L'ACCÈS — à lancer en premier
# ============================================================
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

BASE = Path("/content/drive/MyDrive/Projet_Plantes")

if not BASE.exists():
    print("❌ Projet_Plantes INTROUVABLE dans MyDrive.\n")
    print("Le dossier est en 'Partagés avec moi' : Colab ne le voit pas.")
    print("Corrige en 10 secondes :")
    print("   1. Google Drive → Partagés avec moi")
    print("   2. Clic droit sur 'Projet_Plantes'")
    print("   3. Organiser → Ajouter un raccourci")
    print("   4. Choisis 'Mon Drive' → Ajouter")
    print("   5. Relance cette cellule\n")
    raise SystemExit("Raccourci Drive manquant — voir instructions ci-dessus.")

print("✅ Projet_Plantes accessible")
print("Contenu :", sorted(p.name for p in BASE.iterdir())[:10])

In [ ]:
# ============================================================
# CHARGEMENT DONNEES
# ============================================================

import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns


FEAT_DIR = BASE / "features" / "maladies_raw"


tr = np.load(FEAT_DIR / "train.npz", allow_pickle=True)
val = np.load(FEAT_DIR / "val.npz", allow_pickle=True)
te = np.load(FEAT_DIR / "test.npz", allow_pickle=True)


X_tr_px_maladies, y_tr_str = tr["X"], tr["y"]
X_val_px_maladies, y_val_str = val["X"], val["y"]
X_te_px_maladies, y_te_str = te["X"], te["y"]


# Encodage labels
le_px_maladies = LabelEncoder().fit(y_tr_str)


y_tr_px_maladies = le_px_maladies.transform(y_tr_str)
y_val_px_maladies = le_px_maladies.transform(y_val_str)
y_te_px_maladies = le_px_maladies.transform(y_te_str)



print("Train X :", X_tr_px_maladies.shape)
print("Val X   :", X_val_px_maladies.shape)
print("Test X  :", X_te_px_maladies.shape)

print("Nombre classes :", len(le_px_maladies.classes_))
print(le_px_maladies.classes_[:10])





In [ ]:
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV


rf_pixels = RandomForestClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)


param_grid = {
    "n_estimators": [50, 100],
    "max_features": ["sqrt"],
    "max_depth": [10, 20]
}


grid_pixels = GridSearchCV(
    estimator=rf_pixels,
    param_grid=param_grid,
    cv=3,
    scoring="f1_weighted",
    verbose=2,
    n_jobs=-1
)


grid_pixels.fit(
    X_tr_px_maladies,
    y_tr_px_maladies
)


best_rf_pixels = grid_pixels.best_estimator_


print("Best params :", grid_pixels.best_params_)
print("Best CV score :", grid_pixels.best_score_)


In [ ]:

# ============================================================
# PREDICTIONS TRAIN / VAL / TEST
# ============================================================

y_train_pred = best_rf_pixels.predict(X_tr_px_maladies)
y_val_pred   = best_rf_pixels.predict(X_val_px_maladies)
y_test_pred  = best_rf_pixels.predict(X_te_px_maladies)



In [ ]:
# ============================================================
# SCORES
# ============================================================

print("\n========== ACCURACY ==========")

print(
    "TRAIN :",
    accuracy_score(y_tr_px_maladies, y_train_pred)
)

print(
    "VAL   :",
    accuracy_score(y_val_px_maladies, y_val_pred)
)

print(
    "TEST  :",
    accuracy_score(y_te_px_maladies, y_test_pred)
)



print("\n========== F1 WEIGHTED ==========")

print(
    "TRAIN :",
    f1_score(y_tr_px_maladies, y_train_pred, average="weighted")
)

print(
    "VAL   :",
    f1_score(y_val_px_maladies, y_val_pred, average="weighted")
)

print(
    "TEST  :",
    f1_score(y_te_px_maladies, y_test_pred, average="weighted")
)

In [ ]:

# ============================================================
# CLASSIFICATION REPORT TEST
# ============================================================

print("\n=== CLASSIFICATION REPORT TEST ===\n")

print(
    classification_report(
        y_te_px_maladies,
        y_test_pred,
        target_names=le_px_maladies.classes_))



In [ ]:
# ============================================================
# MATRICE DE CONFUSION TEST
# ============================================================

cm = confusion_matrix(
    y_te_px_maladies,
    y_test_pred
)


plt.figure(figsize=(12,10))


sns.heatmap(
    cm,
    annot=False,
    fmt="d",
    cmap="Blues",
    xticklabels=le_px_maladies.classes_,
    yticklabels=le_px_maladies.classes_
)


plt.xlabel("Prédit")
plt.ylabel("Réel")
plt.title("Matrice de confusion - Random Forest pixels - Maladies")

plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

## Ranfom Forest sur HOG

In [ ]:

# ============================================================
# CHARGEMENT HOG
# ============================================================

FEAT_DIR_HOG = BASE / "features" / "maladies_hog"


tr = np.load(FEAT_DIR_HOG / "train.npz", allow_pickle=True)
val = np.load(FEAT_DIR_HOG / "val.npz", allow_pickle=True)
te = np.load(FEAT_DIR_HOG / "test.npz", allow_pickle=True)


X_tr_hog_maladies, y_tr_str = tr["X"], tr["y"]
X_val_hog_maladies, y_val_str = val["X"], val["y"]
X_te_hog_maladies, y_te_str = te["X"], te["y"]



# Encodage
le_hog_maladies = LabelEncoder().fit(y_tr_str)


y_tr_hog_maladies = le_hog_maladies.transform(y_tr_str)
y_val_hog_maladies = le_hog_maladies.transform(y_val_str)
y_te_hog_maladies = le_hog_maladies.transform(y_te_str)



print("Train HOG :", X_tr_hog_maladies.shape)
print("Val HOG   :", X_val_hog_maladies.shape)
print("Test HOG  :", X_te_hog_maladies.shape)

print("Classes :", len(le_hog_maladies.classes_))


In [ ]:
# ============================================================
# RANDOM FOREST HOG
# ============================================================

rf_hog = RandomForestClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)


param_grid_hog = {
    "n_estimators": [50,100],
    "max_features": ["sqrt"],
    "max_depth": [10,20]
}


grid_hog = GridSearchCV(
    rf_hog,
    param_grid_hog,
    cv=3,
    scoring="f1_weighted",
    verbose=2,
    n_jobs=-1
)


grid_hog.fit(
    X_tr_hog_maladies,
    y_tr_hog_maladies
)


best_rf_hog = grid_hog.best_estimator_


print("Best params :", grid_hog.best_params_)
print("Best CV score :", grid_hog.best_score_)

In [ ]:
# ============================================================
# PREDICTIONS TRAIN / VAL / TEST
# ============================================================

y_train_pred_hog = best_rf_hog.predict(X_tr_hog_maladies)
y_val_pred_hog   = best_rf_hog.predict(X_val_hog_maladies)
y_test_pred_hog  = best_rf_hog.predict(X_te_hog_maladies)

In [ ]:
# ============================================================
# SCORES HOG
# ============================================================

print("\n========== ACCURACY HOG ==========")

print(
    "TRAIN :",
    accuracy_score(y_tr_hog_maladies,y_train_pred_hog)
)

print(
    "VAL   :",
    accuracy_score(y_val_hog_maladies,y_val_pred_hog)
)

print(
    "TEST  :",
    accuracy_score(y_te_hog_maladies,y_test_pred_hog)
)



print("\n========== F1 WEIGHTED HOG ==========")

print(
    "TRAIN :",
    f1_score(
        y_tr_hog_maladies,
        y_train_pred_hog,
        average="weighted"
    )
)

print(
    "VAL   :",
    f1_score(
        y_val_hog_maladies,
        y_val_pred_hog,
        average="weighted"
    )
)

print(
    "TEST  :",
    f1_score(
        y_te_hog_maladies,
        y_test_pred_hog,
        average="weighted"
    )
)

In [ ]:
# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("\n=== CLASSIFICATION REPORT TEST HOG ===\n")


print(
    classification_report(
        y_te_hog_maladies,
        y_test_pred_hog,
        target_names=le_hog_maladies.classes_))


In [ ]:
# ============================================================
# MATRICE DE CONFUSION
# ============================================================

cm_hog = confusion_matrix(
    y_te_hog_maladies,
    y_test_pred_hog)


plt.figure(figsize=(12,10))

sns.heatmap(
    cm_hog,
    annot=False,
    fmt="d",
    cmap="Blues",
    xticklabels=le_hog_maladies.classes_,
    yticklabels=le_hog_maladies.classes_)

plt.xlabel("Prédit")
plt.ylabel("Réel")
plt.title("Matrice de confusion - Random Forest HOG - Maladies")

plt.xticks(rotation=45,ha="right")
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SCORES PIXELS
# ============================================================

acc_pixels = accuracy_score(
    y_te_px_maladies,
    y_test_pred
)

f1_pixels = f1_score(
    y_te_px_maladies,
    y_test_pred,
    average="weighted"
)

# ============================================================
# SCORES HOG
# ============================================================

acc_hog = accuracy_score(
    y_te_hog_maladies,
    y_test_pred_hog
)

f1_hog = f1_score(
    y_te_hog_maladies,
    y_test_pred_hog,
    average="weighted"
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Données
methods = ["Pixels", "HOG"]

accuracy = [
    acc_pixels,
    acc_hog]

f1 = [
    f1_pixels,
    f1_hog]

# Position des barres
x = np.arange(len(methods))
width = 0.35

plt.figure(figsize=(8,5))

# Accuracy en bleu
bars1 = plt.bar(
    x - width/2,
    accuracy,
    width,
    color="steelblue",
    label="Accuracy")

# F1 en orange
bars2 = plt.bar(
    x + width/2,
    f1,
    width,
    color="lightblue",
    label="F1 weighted")

# Affichage des valeurs au-dessus des barres
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width()/2,
            height + 0.01,
            f"{height:.3f}",
            ha="center",
            fontsize=10)

plt.style.use("default")

plt.xticks(x, methods)
plt.ylabel("Score")
plt.ylim(0, 1)

plt.title("Comparaison Random Forest - Pixels vs HOG")

plt.legend()
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()